# Comprensión de los datos y EDA

Análisis exploratorio del dataset de créditos.
Variable objetivo: `Pago_atiempo`.

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None) # Muesta todas las columnas
sns.set_theme(style="whitegrid") # Tema de visualización de gráficos

RUTA_PROYECTO = Path.cwd().parent.parent # Navega a carpeta principal (2 niveles arriba)
df = pd.read_csv(RUTA_PROYECTO / "Base_de_datos.csv") # Carga el CSV

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}") # Imprime dimensiones

Filas: 10763 | Columnas: 23


## Estructura general

In [48]:
df.info() # Mostramos tipos de datos y valores nulos

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tipo_credito                   10763 non-null  int64  
 1   fecha_prestamo                 10763 non-null  str    
 2   capital_prestado               10763 non-null  float64
 3   plazo_meses                    10763 non-null  int64  
 4   edad_cliente                   10763 non-null  int64  
 5   tipo_laboral                   10763 non-null  str    
 6   salario_cliente                10763 non-null  int64  
 7   total_otros_prestamos          10763 non-null  int64  
 8   cuota_pactada                  10763 non-null  int64  
 9   puntaje                        10763 non-null  float64
 10  puntaje_datacredito            10757 non-null  float64
 11  cant_creditosvigentes          10763 non-null  int64  
 12  huella_consulta                10763 non-null  int64  
 1

## Resumen de todas las variables

In [49]:
resumen = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),              # Tipo de dato
    "no_nulos": df.notna().sum(),                       # Cantidad de no nulos
    "nulos": df.isna().sum(),                           # Cantidad de nulos
    "pct_nulos": (df.isna().mean() * 100).round(1),     # Porcentaje de nulos
    "valores_unicos": df.nunique()                      # Cuántos valores diferentes hay
})

resumen.sort_values("pct_nulos", ascending=False)  # Ordena por % de nulos (mayor a menor)

,tipo_pandas,no_nulos,nulos,pct_nulos,valores_unicos
promedio_ingresos_datacredito,float64,7833,2930,27.2,5309
tendencia_ingresos,str,7831,2932,27.2,46
saldo_mora_codeudor,float64,10173,590,5.5,4
saldo_principal,float64,10358,405,3.8,8647
saldo_mora,float64,10607,156,1.4,55
saldo_total,float64,10607,156,1.4,8858
puntaje_datacredito,float64,10757,6,0.1,315
salario_cliente,int64,10763,0,0.0,1385
tipo_laboral,str,10763,0,0.0,2
edad_cliente,int64,10763,0,0.0,54


### Observaciones

**Huecos.** Cinco columnas tienen faltantes, en dos escalas muy distintas:

- `promedio_ingresos_datacredito` (2.930, 27,2%) y `tendencia_ingresos`(2.932, 27,2%) — casi tres de cada diez registros. 
  
  Difieren en dos filas, así que probablemente falten en los mismos créditos.

- `saldo_mora_codeudor` (590), `saldo_principal` (405), `saldo_mora` y `saldo_total` (156 cada una) — manejables.

**Poca variedad.**

- `saldo_mora_codeudor`: 4 valores distintos en 10.173 registros.

- `saldo_mora`: 55 valores, frente a los 8.858 de `saldo_total` sobre la misma cantidad de filas. Dos columnas de montos que se comportan muy distinto.

- `tipo_laboral`: 2 valores, es dicotómica.

- `tipo_credito`: 6 valores y figura como `int64`. Son códigos de categoría, no cantidades — su promedio no significa nada.

**Demasiada variedad.** `tendencia_ingresos` tiene 46 valores únicos cuando debería tener 3. 

Se coló algo que no corresponde.

**La fecha.** `fecha_prestamo` tiene 10.758 valores sobre 10.763 registros: incluye la hora, así que no se repite. 

Funciona como identificador, no como predictor. Para usarla hay que derivar el mes o el año.

## Análisis de anomalías

In [50]:
#Inspeccionamos las columnas problematicas detectadas en el resumen anterior.
print("--- tipo_laboral ---")
print(df["tipo_laboral"].value_counts())            # Cuenta frecuencias

print("\n--- tipo_credito ---")
print(df["tipo_credito"].value_counts())            # Idem

print("\n--- saldo_mora_codeudor ---")
print(df["saldo_mora_codeudor"].value_counts(dropna=False))  # Incluye nulos

print("\n--- saldo_mora (top 10) ---")
print(df["saldo_mora"].value_counts().head(10))    # Top 10 valores más frecuentes

--- tipo_laboral ---
tipo_laboral
Empleado         6754
Independiente    4009
Name: count, dtype: int64

--- tipo_credito ---
tipo_credito
4     7747
9     2876
10     116
6       21
7        2
68       1
Name: count, dtype: int64

--- saldo_mora_codeudor ---
saldo_mora_codeudor
0.0       10170
NaN         590
2145.0        1
30.0          1
470.0         1
Name: count, dtype: int64

--- saldo_mora (top 10) ---
saldo_mora
0.0       10552
30.0          2
132.0         1
366.0         1
5721.0        1
67.0          1
2145.0        1
238.0         1
63.0          1
4379.0        1
Name: count, dtype: int64


### Observaciones

- **`saldo_mora_codeudor`**: 10.170 de 10.173 registros son 0 (99,97%). 
Solo tres tienen otro valor. Con esa variabilidad casi no distingue entre clientes, aunque conviene evaluarlo antes de descartarla.

- **`saldo_mora`**: 10.552 de 10.607 son 0 (99,5%). 
Además del mismo problema, arrastra uno peor: un saldo vencido es consecuencia del incumplimiento, no un antecedente. 
Cuando llega una solicitud nueva ese dato todavía no existe, usarla sería darle al modelo información del futuro, así que se excluye.

- **`tipo_credito`**: los tipos 4 (7.747) y 9 (2.876) concentran el 99%.
Los tipos 10, 6 y 7 suman 139 registros y el tipo 68 aparece una sola vez, rompiendo la secuencia de los demás (4, 6, 7, 9, 10).
Una categoría con uno o dos casos puede quedar entera de un lado al particionar los datos.

- **`tipo_laboral`**: dicotómica (Empleado 6.754, Independiente 4.009), sin nulos y bien balanceada. 
La más limpia del dataset.

### Relación entre saldo_mora y saldo_mora_codeudor

In [51]:
# ¿Coinciden los casos con mora en titular y codeudor?
mora_titular = df["saldo_mora"] > 0                              # Crear booleano: ¿hay mora?
mora_codeudor = df["saldo_mora_codeudor"] > 0                    # Idem para codeudor

print("Titular con mora:", mora_titular.sum())                   # Contar True (mora)
print("Codeudor con mora:", mora_codeudor.sum())                 # Idem
print("Ambos con mora:", (mora_titular & mora_codeudor).sum())   # Contar registros con ambos en mora

# ¿Los nulos también coinciden?
print("\nNulos saldo_mora:", df["saldo_mora"].isna().sum())       # Contar NaN
print("Nulos saldo_mora_codeudor:", df["saldo_mora_codeudor"].isna().sum())
print("Nulos en ambas:", (df["saldo_mora"].isna() & df["saldo_mora_codeudor"].isna()).sum())

Titular con mora: 55
Codeudor con mora: 3
Ambos con mora: 3

Nulos saldo_mora: 156
Nulos saldo_mora_codeudor: 590
Nulos en ambas: 156


Los 3 casos con mora del codeudor son un subconjunto exacto de los 55 del titular: no hay ni un registro donde el codeudor esté en mora y el titular al día.

Tiene sentido — al codeudor se le reclama recién tras el incumplimiento del titular. Así que esta columna no agrega nada: sus 3 casos ya están contados en la otra.

Un detalle aparte: los 156 nulos de `saldo_mora` están contenidos en los 590 de `saldo_mora_codeudor`. Las 434 filas restantes serían créditos sin codeudor, donde el 0 no distingue entre "no debe" y "no hay codeudor".

### Relación entre saldo_total y saldo_principal

In [52]:
# saldo_total: saldo de todas las obligaciones reportadas
# saldo_principal: solo el capital pendiente ("lo que debe, vencido o no")
saldos = df[["saldo_total", "saldo_principal"]].dropna()

iguales = (saldos["saldo_total"] == saldos["saldo_principal"]).sum()
print(f"Registros con ambos saldos: {len(saldos)}")
print(f"Valores idénticos: {iguales} ({iguales/len(saldos)*100:.1f}%)")
print(f"Correlación: {saldos.corr().iloc[0,1]:.4f}")

Registros con ambos saldos: 10358
Valores idénticos: 8658 (83.6%)
Correlación: 0.7347


### Observaciones

De los 10.358 registros con ambos saldos informados, el 83,6% tiene valores idénticos, pero la correlación es 0,7347.

Esa aparente contradicción se explica por el 16% restante: ahí la diferencia entre ambos saldos es grande, y la correlación es sensible a los valores extremos.

La lectura: la mayoría de los clientes debe únicamente capital, y ahí ambas columnas coinciden. En el 16% restante hay un componente adicional que el dataset no permite identificar.

No son redundantes, entonces se conservan las dos — con 0,73 tampoco hay problema de multicolinealidad. La diferencia entre ambas queda anotada como posible atributo derivado.

### Inspeccionar tendencia_ingresos

In [53]:
print(df["tendencia_ingresos"].value_counts(dropna=False).head(15))  # Top 15 valores (incluyendo NaN)
print()
print("Total valores únicos:", df["tendencia_ingresos"].nunique())    # Contar valores distintos

tendencia_ingresos
Creciente      5294
NaN            2932
Decreciente    1291
Estable        1188
0                 7
8315              6
1000000           4
9147              2
158042            1
3978              1
168750            1
-28589            1
-566272           1
24702             1
31837             1
Name: count, dtype: int64

Total valores únicos: 46


### Observaciones

La columna tiene las tres categorías esperadas 
—Creciente (5.294), Decreciente (1.291), Estable (1.188)— más 2.932 nulos y **58 registros con números adentro**.

Esos números incluyen negativos (`-28589`, `-566272`) y montos altos (`1000000`, `158042`), en la misma escala que `salario_cliente` y `promedio_ingresos_datacredito`.

**Hipótesis, no verificable con los datos:** serían la variación de ingresos expresada como monto en lugar de su categoría. El signo encaja, pero no hay forma de confirmarlo.

**Por eso no se decide todavía.** Recodificar por signo se apoyaría entero en esa hipótesis. Las alternativas para la etapa de preparación:

1. Recodificar por signo (positivo → Creciente, negativo → Decreciente, cero → Estable), asumiendo la hipótesis.
2. Convertirlos a nulo, tratándolos como inválidos.
3. Darles categoría propia (`valor_anomalo`), preservando la condición sin interpretarla.
4. Excluir la variable, que ya tiene 27,2% de faltantes.

Son 58 registros sobre 10.763 (0,5%), así que el impacto de cualquiera de las cuatro es acotado.

## Duplicados

In [54]:
print("Duplicados exactos:", df.duplicated().sum())  # Cuenta filas idénticas

Duplicados exactos: 0


## Estadísticos descriptivos

In [55]:
#Muestra resumen estadístico de todas las columnas numéricas.
df.describe().T  # Transpone para ver mejor: min, max, media, mediana, cuartiles, etc.

,count,mean,std,min,25%,50%,75%,max
tipo_credito,10763.0,5.411131e+00,2.338279e+00,4.00000,4.000000e+00,4.000000e+00,9.000000e+00,6.800000e+01
capital_prestado,10763.0,2.434315e+06,1.909643e+06,360000.00000,1.224831e+06,1.921920e+06,3.084840e+06,4.144415e+07
plazo_meses,10763.0,1.057558e+01,6.632082e+00,2.00000,6.000000e+00,1.000000e+01,1.200000e+01,9.000000e+01
edad_cliente,10763.0,4.394862e+01,1.506088e+01,19.00000,3.300000e+01,4.200000e+01,5.300000e+01,1.230000e+02
salario_cliente,10763.0,1.721643e+07,3.554767e+08,0.00000,2.000000e+06,3.000000e+06,4.875808e+06,2.200000e+10
total_otros_prestamos,10763.0,6.238870e+06,1.184183e+08,0.00000,5.000000e+05,1.000000e+06,2.000000e+06,6.787675e+09
cuota_pactada,10763.0,2.436174e+05,2.104937e+05,23944.00000,1.210415e+05,1.828630e+05,2.878335e+05,3.816752e+06
puntaje,10763.0,9.117004e+01,1.646544e+01,-38.00999,9.522779e+01,9.522779e+01,9.522779e+01,9.522779e+01
puntaje_datacredito,10757.0,7.807908e+02,1.048780e+02,-7.00000,7.570000e+02,7.910000e+02,8.250000e+02,9.990000e+02
cant_creditosvigentes,10763.0,5.726749e+00,3.977162e+00,0.00000,3.000000e+00,5.000000e+00,8.000000e+00,6.200000e+01


### Detección de anomalías y valores extremos

In [56]:
print("puntaje == 95.227787:", (df["puntaje"].round(6) == 95.227787).sum())  # Contar registros con ese valor constante
print("puntaje negativo:", (df["puntaje"] < 0).sum())                          # Contar valores inválidos (negativos)
print()
print("salario == 0:", (df["salario_cliente"] == 0).sum())                      # Contar salarios 0 (posibles datos faltantes)
print("salario > 50 millones:", (df["salario_cliente"] > 50_000_000).sum())     # Contar valores extremadamente altos
print()
print("edad > 90:", (df["edad_cliente"] > 90).sum())                            # Contar edades improbables
print("puntaje_datacredito < 0:", (df["puntaje_datacredito"] < 0).sum())        # Contar valores fuera de rango válido

puntaje == 95.227787: 9407
puntaje negativo: 135

salario == 0: 24
salario > 50 millones: 97

edad > 90: 150
puntaje_datacredito < 0: 1


### Observaciones sobre los estadísticos descriptivos

- **`puntaje`**: 
9.407 de 10.763 registros (87,4%) comparten el valor `95.227787`, que además es el máximo. 
Los percentiles 25, 50 y 75 coincidenen ese número.
Hay también 135 valores negativos, inválidos para un score.
  Se analiza aparte en la sección siguiente.

- **`salario_cliente`**: 
la media (17,2 M) quintuplica a la mediana (3,0 M), con desviación estándar de 355 M. 
Hay 97 registros por encima de 50 millones —el máximo llega a 22.000 millones— y 24 con salario 0, que para un titular
  de crédito no tiene sentido: probablemente sea un faltante escrito como cero.

- **`total_otros_prestamos`**: 
misma asimetría (media 6,2 M vs. mediana 1,0 M, máximo 6.787 M).

- **`edad_cliente`**: 
150 registros superan los 90 años, con máximo de 123.

- **`puntaje_datacredito`**:
un registro en -7, fuera del rango del score.

- **`Pago_atiempo`**:
media de 0,9525, o sea 95,25% de clase mayoritaria — solo 511 impagos sobre 10.763. Un modelo que prediga "paga" para todos acierta el 95% sin detectar un solo caso de riesgo. 
Por eso la evaluación tiene que apoyarse en recall y F1, nunca en accuracy.

### Relación de `puntaje` con la variable objetivo

La concentración de valores en `puntaje` motiva verificar si el valor constante
se asocia al resultado del crédito.

In [57]:
# Verifica si el valor constante de puntaje separa perfectamente las clases (predictor perfecto = problema)

puntaje_constante = df["puntaje"].round(6) == 95.227787              # Crear booleano para ese valor

print("Frecuencias:")
print(pd.crosstab(puntaje_constante, df["Pago_atiempo"]))            # Tabla cruzada: puntaje vs pago

print("\nPorcentaje por fila:")
print((pd.crosstab(puntaje_constante, df["Pago_atiempo"], normalize="index") * 100).round(2))  # % por fila

print("\nDistribución de puntaje por clase:")
print(df.groupby("Pago_atiempo")["puntaje"].describe().round(2))     # Estadísticas de puntaje por clase objetivo

Frecuencias:
Pago_atiempo    0     1
puntaje                
False         511   845
True            0  9407

Porcentaje por fila:
Pago_atiempo      0       1
puntaje                    
False         37.68   62.32
True           0.00  100.00

Distribución de puntaje por clase:
                count   mean    std    min    25%    50%    75%    max
Pago_atiempo                                                          
0               511.0  23.09  26.07 -38.01  -3.59  25.42  47.61  62.67
1             10252.0  94.56   2.87  63.81  95.23  95.23  95.23  95.23


### Observaciones

**El valor constante separa perfectamente las clases.** 
Los 9.407 registros con `95.227787` pagaron a tiempo sin excepción. 
Los 511 impagos están íntegramente entre los 1.356 con valor distinto, donde el incumplimiento sube al 37,68%.

**Y los rangos no se solapan.** 
El máximo de `puntaje` entre los impagos es 62,67; el mínimo entre los que pagaron es 63,81. 
Un umbral en torno a 63 clasifica correctamente el 100% de los registros.

Ningún dato real predice el futuro sin equivocarse. 
Una separación perfecta significa que el valor se determinó después de conocer el resultado, o que directamente lo codifica. 
Si entra al entrenamiento, el modelo va a dar métricas casi perfectas en las pruebas y a fallar con clientes nuevos, porque
ese puntaje no existe al momento de la solicitud.

**Se excluye.** 
Y vale notar que la conclusión es la opuesta a la primera impresión: el problema de esta columna no es la falta de información sino su exceso.

## Variable temporal

In [58]:
# Analiza el rango temporal del dataset y distribución mensual.

fechas = pd.to_datetime(df["fecha_prestamo"])        # Convertir a datetime

print("Desde:", fechas.min())                         # Primera fecha del dataset
print("Hasta:", fechas.max())                         # Última fecha
print("Rango:", (fechas.max() - fechas.min()).days, "días")  # Duración total
print()
print(fechas.dt.to_period("M").value_counts().sort_index())  # Agrupar por mes y contar

Desde: 2024-11-26 09:17:04
Hasta: 2026-04-26 18:43:52
Rango: 516 días

fecha_prestamo
2024-11     186
2024-12    1216
2025-01    1917
2025-02    1094
2025-03    1115
2025-04    1102
2025-05    1083
2025-06     665
2025-07     644
2025-08     555
2025-09     361
2025-10     231
2025-11     166
2025-12     174
2026-01     127
2026-02      76
2026-03      40
2026-04      11
Freq: M, Name: count, dtype: int64


### Cruce mensual con otras variables

In [59]:
#Tabla resumen por mes para ver patrones temporales y qué tan completos están los datos

df_temp = df.copy()                                   # Copia para no modificar original
df_temp["mes"] = pd.to_datetime(df_temp["fecha_prestamo"]).dt.to_period("M")  # Extraer mes

resumen_mes = df_temp.groupby("mes").agg(            # Agrupar por mes y calcular:
    creditos=("Pago_atiempo", "size"),               # - Total de créditos
    pct_impago=("Pago_atiempo", lambda x: (1 - x.mean()) * 100),  # - % de impago
    pct_nulos_ingresos=("promedio_ingresos_datacredito", lambda x: x.isna().mean() * 100),  # - % de nulos en ingresos
    pct_nulos_saldo=("saldo_principal", lambda x: x.isna().mean() * 100),  # - % de nulos en saldo
    salario_medio=("salario_cliente", "median"),     # - Salario mediano
    edad_media=("edad_cliente", "mean"),             # - Edad promedio
).round(1)

resumen_mes

,creditos,pct_impago,pct_nulos_ingresos,pct_nulos_saldo,salario_medio,edad_media
mes,,,,,,
2024-11,186,7.0,32.8,6.5,2500000.0,42.0
2024-12,1216,6.1,31.9,3.5,2800000.0,43.1
2025-01,1917,4.5,26.2,2.5,3000000.0,44.0
2025-02,1094,5.9,23.7,3.7,3000000.0,44.2
2025-03,1115,5.1,26.1,3.8,3100000.0,44.2
2025-04,1102,6.3,27.0,3.4,3100000.0,44.4
2025-05,1083,4.3,27.3,5.2,3500000.0,45.1
2025-06,665,3.8,26.2,5.6,3500000.0,45.0
2025-07,644,3.1,35.6,4.7,3000000.0,44.3


### Observaciones

El dataset abarca 516 días, del 26/11/2024 al 26/04/2026. 

El volumen mensual cae de forma sostenida: 
**1.917 créditos en enero de 2025 contra 11 en abril de 2026**. 
Ninguna financiera real pierde el 99% de su actividad en línea recta, así que probablemente sea un efecto de cómo se armó el archivo.

Probamos dos explicaciones y ninguna encaja:

1. **Maduración de los créditos.** 
Si los meses recientes trajeran solo créditos de plazo corto —los únicos ya vencidos— el plazo promedio debería bajar. 
Pasalo contrario: 
baja hasta 9,2 meses en junio de 2025 y después sube hasta 16,5 en enero de 2026. 
Esto no descarta del todo un efecto de maduración:
comprobarlo requeriría la fecha de extracción, que el dataset no incluye.

2. **Corte abrupto en la extracción.** 
Un corte daría actividad normal hasta cierto día y nada después. 
Los registros siguen apareciendo hasta el final, cada vez más espaciados:
en enero de 2026 hay créditos casi todos los días,en abril quedan nueve días con uno cada uno. 
Descarta el corte abrupto, pero no una carga parcial, un ETL incompleto o un filtro desconocido — cualquiera de esos produce justamente un descenso gradual.

**Otros datos de la tabla mensual.** 
La tasa de impago baja del 7,0% en noviembre de 2024 al 1,7% en octubre y diciembre de 2025. 
Una explicación posible, no verificada acá, es que los créditos recientes hayan tenido menos tiempo para caer en mora. 
Los últimos meses (76, 40 y 11 registros) no son interpretables: 
el 9,1% de abril de 2026 es un único caso.

El perfil del cliente se mantiene estable en todo el período —salario mediano entre 2,5 y 3,5 M, edad promedio entre 40 y 45 años—, lo que descarta un cambio en la política de otorgamiento.

No hay información que permita determinar la causa ni descartar problemas de cobertura. 
Queda anotado para definir la estrategia de partición en el modelado.

## Patrón de los valores faltantes

In [60]:
# Verifica si los datos faltantes se concentran en algún período temporal específico.

df_temp = df.copy()                                                          # Copia del dataframe
df_temp["fecha"] = pd.to_datetime(df_temp["fecha_prestamo"])                 # Convertir fecha a datetime
df_temp["sin_ingresos"] = df_temp["promedio_ingresos_datacredito"].isna()    # Crear booleano: ¿falta dato?

# Comparar la fecha promedio de los que tienen dato vs los que no
print("Con ingresos - fecha mediana:", df_temp[~df_temp["sin_ingresos"]]["fecha"].median())       # Mediana de fechas CON dato
print("Sin ingresos - fecha mediana:", df_temp[df_temp["sin_ingresos"]]["fecha"].median())        # Mediana de fechas SIN dato

Con ingresos - fecha mediana: 2025-03-27 15:48:51
Sin ingresos - fecha mediana: 2025-03-28 12:08:57.500000


### Asociaciones de los datos faltantes

Los nulos no se distribuyen uniformemente. Ahora analizamos cómo se asocian con características del cliente y del riesgo de incumplimiento.

In [61]:
df_temp["sin_ingresos"] = df_temp["promedio_ingresos_datacredito"].isna()

# Proporción de faltantes por tipo de empleo
print(df_temp.groupby("tipo_laboral")["sin_ingresos"].mean().round(3))       # % de faltantes por tipo_laboral
print()
# Proporción de faltantes por tipo de crédito
print(df_temp.groupby("tipo_credito")["sin_ingresos"].mean().round(3))       # % de faltantes por tipo_credito
print()
# Tasa de impago según si hay dato de ingresos
print("Impago con ingresos:", (1 - df_temp[~df_temp["sin_ingresos"]]["Pago_atiempo"].mean()).round(4))    # % impago TIENE dato
print("Impago sin ingresos:", (1 - df_temp[df_temp["sin_ingresos"]]["Pago_atiempo"].mean()).round(4))     # % impago SIN dato

tipo_laboral
Empleado         0.19
Independiente    0.41
Name: sin_ingresos, dtype: float64

tipo_credito
4     0.252
6     0.190
7     0.500
9     0.329
10    0.250
68    0.000
Name: sin_ingresos, dtype: float64

Impago con ingresos: 0.0438
Impago sin ingresos: 0.0573


In [62]:
from scipy.stats import chi2_contingency

tabla = pd.crosstab(df["promedio_ingresos_datacredito"].isna(), df["Pago_atiempo"])
chi2, p_valor, _, _ = chi2_contingency(tabla)

print(tabla)
print(f"\np-valor: {p_valor:.4f}")

Pago_atiempo                     0     1
promedio_ingresos_datacredito           
False                          343  7490
True                           168  2762

p-valor: 0.0038


### Observaciones

**Sin concentración temporal.** 
La fecha mediana de los registros con dato de ingresos (2025-03-27) y la de los que no lo tienen (2025-03-28) difiere en un
día sobre un rango de 516. 
El aparente salto de nulos en los últimos meses —del 16,7% en diciembre de 2025 al 90,9% en abril de 2026— es un espejismo del
bajo volumen: ese 90,9% son 10 registros sobre 11.

**Sí se asocia al tipo laboral.** 
El 41% de los independientes no tiene el dato,contra el 19% de los empleados: más del doble. 
Una explicación posible —no verificable acá— es que verificar ingresos de un trabajador por cuenta propia es más difícil, porque no hay empleador que los reporte. 
Lo que sí queda establecido es más acotado: la ausencia no se reparte de manera uniforme entre los grupos observados. 
Determinar el mecanismo completo excede esta etapa.

**Y se asocia al impago.** 
Los registros sin dato fallan un 5,73% frente al 4,38% de los que sí lo tienen. 
Como la diferencia parecía chica, se midió con chi-cuadrado: 
**p-valor de 0,0038**, así que no se explica por azar. 
Igual la magnitud es moderada (1,35 puntos), de modo que queda establecida la asociación,no su peso como predictor.

Sobre `tipo_credito`, los faltantes van del 19% al 50%, pero solo los tipos 4 y 9 tienen volumen para sostener la comparación (25% y 33%) — y esa diferencia es coherente con el efecto del tipo laboral.

**Implicancia.** 
Al imputar `promedio_ingresos_datacredito` hay que agregar un indicador binario que preserve cuáles estaban vacíos.
Si se rellena y se olvida, esa señal se pierde.

## Limitaciones del dataset

No hay documentación de las variables, así que la interpretación de `tipo_credito`, `huella_consulta` o `puntaje` se apoya en convenciones del dominio y en lo que muestran los propios datos.

Dos puntos quedan abiertos:

- **`puntaje`**: 
no se puede determinar en qué momento del proceso se calcula.
La evidencia empírica —separación perfecta de las clases— alcanza igual para excluirla.

- **`tendencia_ingresos`**: 
no se puede determinar el origen de los 58 valores numéricos ni cuál es el tratamiento correcto.